# Detección de Caries con YOLOv8
**Modelo:** YOLOv8s (GPU T4 gratuita)  
**Clases:** caries leve, caries moderada, caries severa, sin caries

> Antes de ejecutar: activar GPU en Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU

## 0. Configuración del experimento
> **Cambia solo esta celda entre experimentos**

In [ ]:
# ─── EDITAR AQUÍ ANTES DE CADA EXPERIMENTO ───────────────────────────────────
DATASET_VERSION  = 'v3'          # 'v1', 'v2' o 'v3'
DATASET_FOLDER   = f'Data_Caries.{DATASET_VERSION}i.yolov11'
EXPERIMENT_NAME  = f'yolov8s_{DATASET_VERSION}'
# ─────────────────────────────────────────────────────────────────────────────

print(f'Experimento : {EXPERIMENT_NAME}')
print(f'Dataset     : {DATASET_FOLDER}')

## 1. Instalar dependencias

In [ ]:
!pip install ultralytics -q
import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_PATH   = '/content/drive/MyDrive/proyecto-tesis'
DATASET_PATH = f'{DRIVE_PATH}/{DATASET_FOLDER}'
RESULTS_PATH = f'{DRIVE_PATH}/Resultados_Pruebas/{EXPERIMENT_NAME}'

os.makedirs(f'{DRIVE_PATH}/Resultados_Pruebas', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/models', exist_ok=True)

if os.path.exists(DATASET_PATH):
    print(f'Dataset encontrado: {DATASET_FOLDER}')
else:
    print(f'ERROR: No se encontró {DATASET_FOLDER} en Drive')

## 3. Configurar rutas del dataset

In [ ]:
import yaml
from pathlib import Path

data_yaml = Path(DATASET_PATH) / 'data.yaml'

with open(data_yaml) as f:
    data_config = yaml.safe_load(f)

data_config['train'] = f'{DATASET_PATH}/train/images'
data_config['val']   = f'{DATASET_PATH}/valid/images'
data_config['test']  = f'{DATASET_PATH}/test/images'

with open(data_yaml, 'w') as f:
    yaml.dump(data_config, f, allow_unicode=True)

print('Clases:', data_config['names'])
print('Número de clases:', data_config['nc'])
print('Rutas configuradas correctamente')

## 4. Explorar el dataset

In [ ]:
total = 0
for split in ['train', 'valid', 'test']:
    img_dir = Path(DATASET_PATH) / split / 'images'
    if img_dir.exists():
        count = len(list(img_dir.glob('*')))
        total += count
        print(f'{split}: {count} imágenes')
print(f'Total: {total} imágenes')

## 5. Visualizar muestras del dataset con bounding boxes

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

COLORS = ['#22c55e', '#eab308', '#ef4444', '#3b82f6']

def plot_yolo_sample(img_path, label_path, class_names):
    img = Image.open(img_path)
    w, h = img.size
    fig, ax = plt.subplots(1, figsize=(8, 6))
    ax.imshow(img)
    if label_path.exists():
        with open(label_path) as f:
            for line in f.readlines():
                cls, cx, cy, bw, bh = map(float, line.strip().split())
                x = (cx - bw / 2) * w
                y = (cy - bh / 2) * h
                color = COLORS[int(cls) % len(COLORS)]
                rect = patches.Rectangle((x, y), bw * w, bh * h,
                                         linewidth=2, edgecolor=color, facecolor='none')
                ax.add_patch(rect)
                ax.text(x, y - 5, class_names[int(cls)], color=color, fontsize=9,
                        bbox=dict(facecolor='white', alpha=0.6, pad=1))
    ax.axis('off')
    plt.tight_layout()
    plt.show()

train_imgs = list((Path(DATASET_PATH) / 'train' / 'images').glob('*'))[:4]
for img_path in train_imgs:
    label_path = img_path.parent.parent / 'labels' / (img_path.stem + '.txt')
    plot_yolo_sample(img_path, label_path, data_config['names'])

## 6. Entrenar YOLOv8s

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s.pt')

results = model.train(
    data=str(data_yaml),
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    patience=20,
    project=f'{DRIVE_PATH}/Resultados_Pruebas',
    name=EXPERIMENT_NAME,
    exist_ok=True,
    plots=True,
    save=True,
    cache=True
)

print('Entrenamiento completado')
print(f'Mejor modelo en: {RESULTS_PATH}/weights/best.pt')

## 7. Evaluar el modelo en test set

In [ ]:
from ultralytics import YOLO
import json

best_model_path = f'{RESULTS_PATH}/weights/best.pt'
best_model = YOLO(best_model_path)

metrics = best_model.val(data=str(data_yaml), split='test')

class_names = data_config['names']

print(f'\n=== Métricas Globales — {EXPERIMENT_NAME} ===')
print(f'mAP50:     {metrics.box.map50:.4f}')
print(f'mAP50-95:  {metrics.box.map:.4f}')
print(f'Precision: {metrics.box.mp:.4f}')
print(f'Recall:    {metrics.box.mr:.4f}')
print(f'F1 global: {metrics.box.f1.mean():.4f}')

print(f'\n=== Métricas por Clase ===')
per_class = {}
for i, name in enumerate(class_names):
    p      = float(metrics.box.p[i])
    r      = float(metrics.box.r[i])
    f1     = float(metrics.box.f1[i])
    ap50   = float(metrics.box.ap50[i])
    ap5095 = float(metrics.box.ap[i])
    per_class[name] = {
        'precision': round(p,      4),
        'recall':    round(r,      4),
        'f1':        round(f1,     4),
        'mAP50':     round(ap50,   4),
        'mAP50_95':  round(ap5095, 4),
    }
    print(f'  {name:20s} | P={p:.4f}  R={r:.4f}  F1={f1:.4f}  mAP50={ap50:.4f}')

metrics_dict = {
    'experiment': EXPERIMENT_NAME,
    'dataset':    DATASET_FOLDER,
    'mAP50':      round(metrics.box.map50, 4),
    'mAP50_95':   round(metrics.box.map,   4),
    'precision':  round(metrics.box.mp,    4),
    'recall':     round(metrics.box.mr,    4),
    'f1':         round(float(metrics.box.f1.mean()), 4),
    'per_class':  per_class,
}
metrics_file = f'{RESULTS_PATH}/test_metrics.json'
with open(metrics_file, 'w') as f:
    json.dump(metrics_dict, f, indent=2, ensure_ascii=False)
print(f'\nMétricas guardadas en: {metrics_file}')

## 8. Graficar métricas de entrenamiento

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

results_csv = f'{RESULTS_PATH}/results.csv'
df = pd.read_csv(results_csv)
df.columns = df.columns.str.strip()

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
fig.suptitle(f'Métricas de Entrenamiento — {EXPERIMENT_NAME}', fontsize=14)

metrics_to_plot = [
    ('train/box_loss',       'Train Box Loss'),
    ('train/cls_loss',       'Train Class Loss'),
    ('val/box_loss',         'Val Box Loss'),
    ('metrics/precision(B)', 'Precision'),
    ('metrics/recall(B)',    'Recall'),
    ('metrics/mAP50(B)',     'mAP@50'),
]

for ax, (col, title) in zip(axes.flatten(), metrics_to_plot):
    if col in df.columns:
        ax.plot(df['epoch'], df[col])
        ax.set_title(title)
        ax.set_xlabel('Epoch')
        ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/training_metrics.png', dpi=150)
plt.show()

## 9. Inferencia de prueba en imágenes del test set

In [ ]:
import glob

test_images = glob.glob(f'{DATASET_PATH}/test/images/*')[:4]

fig, axes = plt.subplots(1, len(test_images), figsize=(16, 5))
fig.suptitle(f'Predicciones — {EXPERIMENT_NAME}', fontsize=13)

for ax, img_path in zip(axes, test_images):
    result = best_model.predict(img_path, conf=0.25, verbose=False)[0]
    annotated = result.plot()
    ax.imshow(annotated[:, :, ::-1])
    ax.axis('off')

plt.tight_layout()
plt.savefig(f'{RESULTS_PATH}/sample_predictions.png', dpi=150)
plt.show()

## 10. Guardar modelo final

In [ ]:
import shutil

dest = f'{DRIVE_PATH}/models/best_caries_{DATASET_VERSION}.pt'
shutil.copy(best_model_path, dest)
print(f'Modelo guardado en: {dest}')

## 11. Comparar experimentos
> Ejecutar esta celda solo cuando tengas resultados de V1, V2 y V3

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

experiments = ['yolov8s_v1', 'yolov8s_v2', 'yolov8s_v3']
results_all = []

for exp in experiments:
    metrics_file = Path(f'{DRIVE_PATH}/Resultados_Pruebas/{exp}/test_metrics.json')
    if metrics_file.exists():
        with open(metrics_file) as f:
            results_all.append(json.load(f))
    else:
        print(f'Aún no hay resultados para {exp}')

if len(results_all) < 3:
    print('Entrena los tres experimentos antes de comparar')
else:
    df = pd.DataFrame(results_all)
    print('\n=== Comparación de Experimentos ===')
    print(df[['experiment', 'dataset', 'mAP50', 'mAP50_95', 'precision', 'recall', 'f1']].to_string(index=False))

    fig, axes = plt.subplots(1, 5, figsize=(18, 4))
    fig.suptitle('Comparación V1 vs V2 vs V3', fontsize=13)
    metrics_cols = ['mAP50', 'mAP50_95', 'precision', 'recall', 'f1']
    colors = ['#3b82f6', '#22c55e', '#f97316']

    for ax, metric in zip(axes, metrics_cols):
        bars = ax.bar(df['experiment'], df[metric], color=colors)
        ax.set_title(metric)
        ax.set_ylim(0, 1)
        ax.grid(axis='y', alpha=0.3)
        for bar, val in zip(bars, df[metric]):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f'{val:.3f}', ha='center', fontsize=9)

    plt.tight_layout()
    plt.savefig(f'{DRIVE_PATH}/Resultados_Pruebas/comparacion_v1_v2_v3.png', dpi=150)
    plt.show()